Set Up and Load Data

In [1]:
import pandas as pd
import numpy as np # Useful for data manipulation

# Load the raw data files
drivers_df = pd.read_csv("drivers_raw.csv")
riders_df = pd.read_csv("riders_raw.csv")
rides_df = pd.read_csv("rides_raw.csv")
payments_df = pd.read_csv("payments_raw.csv")

# Define the analysis window (June 2021 – December 2024)
START_DATE = pd.to_datetime('2021-06-01')
END_DATE = pd.to_datetime('2024-12-31 23:59:59')

Cleaning the Rides and Payments Tables (Transactional Data)

In [3]:
# Convert date columns in rides_df
date_cols_rides = ['request_time', 'pickup_time', 'dropoff_time']
for col in date_cols_rides:
    rides_df[col] = pd.to_datetime(rides_df[col], errors='coerce')

# Convert date columns in payments_df
payments_df['paid_date'] = pd.to_datetime(payments_df['paid_date'], errors='coerce')

# FILTER: Keep only rides within the analysis window (June 2021 - Dec 2024)
rides_df_filtered = rides_df[
    (rides_df['pickup_time'] >= START_DATE) &
    (rides_df['pickup_time'] <= END_DATE)
].copy()

Standardize City Names

In [4]:
city_map = {
    'S.F': 'San Francisco', 'SF': 'San Francisco',
    'N.Y': 'New York', 'NY': 'New York',
    'L.A': 'Los Angeles', 'LA': 'Los Angeles'
}

# Apply standardization to pickup and dropoff cities
rides_df_filtered['pickup_city'] = rides_df_filtered['pickup_city'].str.strip().replace(city_map)
rides_df_filtered['dropoff_city'] = rides_df_filtered['dropoff_city'].str.strip().replace(city_map)

Data Validation and Conversion

In [5]:
# 1. Distance Validation (Remove rides with non-positive distance)
rides_df_filtered = rides_df_filtered[rides_df_filtered['distance_km'] > 0]

# 2. Distance Conversion (KM to Miles)
KM_TO_MILES = 1.60934
rides_df_filtered['distance_miles'] = rides_df_filtered['distance_km'] / KM_TO_MILES
rides_df_filtered.drop(columns=['distance_km'], inplace=True) # Drop original column

# 3. Standardize 'status' (to lowercase)
rides_df_filtered['status'] = rides_df_filtered['status'].str.lower().str.strip()

Payment Method Standardization

In [6]:
payments_df['method'] = payments_df['method'].str.lower().str.strip().replace({
    'pay pal': 'PayPal',
    'pay-pal': 'PayPal',
    'card': 'Credit Card',
    'mobile': 'Mobile Money' # Assuming 'mobile' is 'Mobile Money'
})

Merge and Final Cleanup

In [7]:
# Merge Rides and Payments on ride_id
rides_payments_df = pd.merge(
    rides_df_filtered,
    payments_df[['ride_id', 'amount', 'method', 'paid_date']],
    on='ride_id',
    how='left',
    suffixes=('_ride', '_payment')
)

# Rename 'fare' (estimated) and 'amount' (actual paid) for clarity
rides_payments_df.rename(columns={'fare': 'estimated_fare'}, inplace=True)

# Duplicates check (on primary key)
rides_payments_df.drop_duplicates(subset='ride_id', keep='first', inplace=True)

Cleaning Drivers and Riders Tables (Master Data)

-Drivers Table Cleanup

In [8]:
# 1. Date Conversion
drivers_df['signup_date'] = pd.to_datetime(drivers_df['signup_date'], errors='coerce')

# 2. Standardize City Names
drivers_df['city'] = drivers_df['city'].str.strip().replace(city_map)

# 3. Data Validation (Rating)
# Keep only valid ratings between 1.0 and 5.0
drivers_df['rating'] = pd.to_numeric(drivers_df['rating'], errors='coerce')
drivers_df = drivers_df[(drivers_df['rating'] >= 1.0) & (drivers_df['rating'] <= 5.0)].copy()

# 4. Duplicates check
drivers_df.drop_duplicates(subset='driver_id', keep='first', inplace=True)

# 5. Add driver rating to the rides table (needed for Q8 using driver's overall rating)
drivers_ratings_df = drivers_df[['driver_id', 'rating']].rename(columns={'rating': 'driver_static_rating'})
rides_payments_df = pd.merge(rides_payments_df, drivers_ratings_df, on='driver_id', how='left')

Riders Table Cleanup

In [9]:
# 1. Date Conversion
riders_df['signup_date'] = pd.to_datetime(riders_df['signup_date'], errors='coerce')

# 2. Standardize City Names
riders_df['city'] = riders_df['city'].str.strip().replace(city_map)

# 3. Duplicates check
riders_df.drop_duplicates(subset='rider_id', keep='first', inplace=True)

Export Cleaned Files

In [10]:
# Rename columns in Drivers and Riders to match SQL schema from previous response
drivers_df.rename(columns={'name': 'driver_name'}, inplace=True)
riders_df.rename(columns={'name': 'rider_name'}, inplace=True)

# Export the final cleaned datasets
rides_payments_df.to_csv('rides_payments_cleaned.csv', index=False)
drivers_df.to_csv('drivers_cleaned.csv', index=False)
riders_df.to_csv('riders_cleaned.csv', index=False)

print("Data cleaning complete. Files saved as: 'rides_payments_cleaned.csv', 'drivers_cleaned.csv', 'riders_cleaned.csv'")

Data cleaning complete. Files saved as: 'rides_payments_cleaned.csv', 'drivers_cleaned.csv', 'riders_cleaned.csv'
